In [1]:
!uv add youtube-transcript-api

Resolved 158 packages in 17ms
Checked 153 packages in 47ms


In [2]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = 'r0gHbW_MmaQ'

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id)

In [3]:
transcript.snippets[-1]

FetchedTranscriptSnippet(text='you. Thanks. See you.', start=3526.079, duration=4.52)

In [4]:
def format_timestamp(seconds: float) -> str:
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"

In [5]:
format_timestamp(3526.079)

'58:46'

In [6]:
transcript_lines = []

for snippet in transcript.snippets:
    ts = format_timestamp(snippet.start)
    line = f'{ts} {snippet.text}'
    transcript_lines.append(line)

In [7]:
subtitles = '\n'.join(transcript_lines[:10])

In [8]:
print(subtitles[:400])

0:00 Hi everyone, welcome to the second
0:02 iteration of AI engineering boot camp.
0:05 So I renamed the course recently. It
0:08 used to be called AI boot camp but there
0:10 are so many boot camps and AI boot camps
0:12 these days. So I was thinking if there's
0:15 a way to somehow be different from these
0:17 courses and uh maybe you know me by uh
0:21 from courses u from free courses from
0:2


In [9]:
def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)

subtitles = make_subtitles(transcript)

In [10]:
from pathlib import Path

subtitles_file = Path(f'subtitles_{video_id}.txt')
subtitles_file.write_text(subtitles, encoding='utf-8')

56201

In [11]:
from pathlib import Path

subtitles_file = Path(f'subtitles_{video_id}.txt')
subtitles = subtitles_file.read_text(encoding='utf-8')

In [13]:
from google import genai
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())


# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Explain how AI works in a few words"
)
print(response.text)

AI works by learning patterns from data to make predictions or decisions.


In [ ]:
def llm(instructions, prompt, model='gemini-2.5-flash-lite'):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "system_instruction": instructions
        }
    )
    return response.text

In [15]:
instructions = """
Summarize the following youtube transcript and create a timestamped
list of video chapters

The goal is to use this as the description under the YouTube video.
The output should be copy-pastable to youtube, so don't start the
summary with "the transcript provides...". 
Don't start with "Welcome...", "In this video," etc

It should contain 2-3 paragraphs of text.

Don't use any formatting in the response.

The correct name of the course is "AI Engineering Buildcamp" 

After the description, create the chapter list in the following format

<timestamp> <chapter name>

Make sure chapters are at least 3-4 minutes long but not longer than
6 minutes. The chapters should 
correspond to what's actually discussed in the video

The chapter should start where the concept is introduced, and should be very
precise - this is going to be used for youtube, and viewers will use it for
quick navigation around the video
"""

In [16]:
summary = llm(
    instructions=instructions,
    prompt=subtitles,
)

In [17]:
print(summary)

Welcome to the AI Engineering Buildcamp! This course focuses on building AI systems with an engineering mindset, emphasizing best practices and testability. Over nine weeks, you'll dive into foundational LLM concepts, building RAG applications, and developing agents. Week one covers course overview, preparation, LLM foundations, and RAG systems using the Evidently AI documentation as a running example. Week two is a buffer week with optional materials like additional RAG use cases.

Weeks three through six delve into agents, their development with frameworks like Pydantic, testing methodologies (including LLM-based judging), and monitoring deployed systems. The curriculum also includes crucial topics like evaluation techniques, building gold standard datasets, and understanding agent performance. Starting from week seven, the focus shifts entirely to your capstone projects, with an example of an end-to-end project walkthrough. The course also touches upon advanced topics like guardrail

In [18]:
from pydantic import BaseModel, Field

class Chapter(BaseModel):
    timestamp: str
    title: str
    reference_line: str = Field(description="")

class YTSummaryResponse(BaseModel):
    summary: str
    chapters: list[Chapter]

    def print(self):
        print(self.summary)
        print()

        print('Chapters:')

        for c in self.chapters:
            print(f'{c.timestamp} {c.title}')

In [19]:
def llm_structured(instructions, prompt, model='gemini-2.5-flash',output=YTSummaryResponse):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "system_instruction": instructions,
            "response_mime_type": "application/json",
            "response_json_schema": output.model_json_schema()
        }
    )

    result = output.model_validate_json(response.text)
    return result

In [21]:
summary = llm_structured(
    instructions=instructions,
    prompt=subtitles,
    model='gemini-2.5-flash-lite'
)

summary.print()

This video introduces the "AI Engineering Buildcamp," a course focused on building practical AI systems using an engineering mindset. The instructor outlines the course structure, which spans nine weeks, covering foundational AI concepts, building RAG applications, agents, testing, monitoring, and deployment. The course emphasizes hands-on projects, including a capstone project that students work on throughout the program. The instructor also details the communication channels, including Slack, and the use of a course management platform for submissions and peer reviews. Key topics like "learning in public," office hours, environment preparation (GitHub Codespaces and local setup), and the use of AI-assisted development tools are discussed. The curriculum includes optional materials and bonus sections on advanced topics like multi-agent systems and deployment strategies. The course culminates in a demo week for project presentations and a peer review week, followed by an optional hacka

In [22]:
instructions = """
You are given a raw YouTube transcript as plain text. Each transcript line is formatted exactly like:

[LINE_INDEX] TIMESTAMP text...

Example:
[0012] 08:03 So this is the library we're going to um

Your job is to return ONLY valid JSON that matches the provided Pydantic schema (YTSummaryResponse).

PRIMARY GOAL
Create a YouTube description + a chapter list that provides FULL COVERAGE of the video:
- Coverage must start at the FIRST timestamp in the transcript.
- Coverage must end at the LAST timestamp in the transcript.
- There must be NO uncovered gaps between chapters.
  (The end of one chapter is the start of the next chapter.)
- Therefore, the first chapter timestamp MUST equal the first transcript timestamp,
  and the last chapter must extend to the end of the video (last transcript timestamp).

OUTPUT FORMAT (STRICT)
- Return ONLY a single JSON object.
- Do not include any additional keys beyond the schema.
- Do not include markdown, code fences, or extra commentary.

SUMMARY RULES
- Write 2–3 paragraphs of plain text (no bullet points, no headings, no markdown).
- Do NOT start with: "Welcome", "In this video", "Today we will", "The transcript...", or similar framing.
- The tone should fit a YouTube description: clear and informative.
- Use the exact course name: "AI Engineering Buildcamp".
- Do not add links unless they are explicitly present in the transcript.

CHAPTER LIST RULES (FULL COVERAGE REQUIRED)
- Chapters must be in chronological order.
- Chapters must provide FULL COVERAGE from the first to the last timestamp as described above.
- Each chapter should start exactly when a new concept/topic is introduced.
- Aim for chapter length of 3–5 minutes.
- Hard constraints: chapters must be at least 3 minutes long and at most 6 minutes long.
- If strict topic boundaries would cause a chapter to be shorter than 3 minutes,
  merge it with a neighboring topic.
- If a topic runs longer than 6 minutes, split it into multiple chapters at sensible subtopic boundaries.
- Create enough chapters to cover the entire duration; do not stop early.

REFERENCE LINE RULES (CRITICAL — NO HALLUCINATIONS)
- For each chapter you MUST choose exactly ONE anchor line from the transcript and copy it verbatim into `reference_line`.
- `reference_line` MUST match one of the provided transcript lines character-for-character,
  including the [LINE_INDEX] prefix, the timestamp, capitalization, filler words, and punctuation.
- Do NOT paraphrase, rewrite, merge multiple lines, or "clean up" the text.
- If you cannot find an exact transcript line to support a chapter start, do NOT include that chapter.
  Instead, choose a different chapter boundary that DOES have an exact supporting line.

TIMESTAMP RULES (CRITICAL)
- `timestamp` MUST be taken from the anchor line you copied into `reference_line`.
- `timestamp` MUST exactly equal the timestamp that appears inside `reference_line`.
- Do NOT invent timestamps. Do NOT approximate. Do NOT shift by a few seconds.
- The first chapter timestamp MUST equal the first transcript timestamp (full coverage).
- Every next chapter timestamp must be later than the previous chapter timestamp.

CHAPTER TITLE RULES
- `title` should be 2–7 words.
- Make it specific and descriptive (avoid vague titles like "More details" or "Next steps").
- No quotes, emojis, or trailing punctuation.

OPTIONAL CHAPTER SUMMARY
- `chapter_summary` may be null or one concise sentence.
- Keep it factual and specific; do not add new information not supported by the transcript.

FINAL SELF-CHECK (DO THIS BEFORE YOU OUTPUT)
1) Identify the first transcript timestamp and ensure the first chapter uses it.
2) Identify the last transcript timestamp and ensure the last chapter reaches it.
3) Ensure there are no gaps: each chapter starts where the previous one ends (i.e., next chapter timestamp is the boundary).
4) Ensure each chapter duration is 3–6 minutes (target 3–5).
5) Ensure every `reference_line` is an exact copy of a transcript line including [LINE_INDEX].
6) Ensure every `timestamp` exactly matches the timestamp inside `reference_line`.

If you cannot satisfy any rule, adjust chapter boundaries and/or increase the number of chapters.

Return JSON only.
"""

In [23]:
from pydantic import BaseModel, Field

class Chapter(BaseModel):
    timestamp: str = Field(
        description="Chapter start timestamp, must match the timestamp in reference_line, format is H:MM:SS or MM:SS."
    )
    title: str = Field(
        description="5-15 word YouTube chapter title."
    )
    reference_line: str = Field(
        description=(
            "A verbatim transcript line INCLUDING its numeric prefix and timestamp, "
            "copied exactly from the provided subtitles. "
            "Must match one of the provided lines character-for-character."
        )
    )


class YTSummaryResponse(BaseModel):
    summary: str
    chapters: list[Chapter]

    def print(self):
        print(self.summary)
        print("\nChapters:\n")
        for c in self.chapters:
            print(f"{c.timestamp} {c.title}")

In [24]:
def preprocess_subtitles(subtitles: str) -> str:
    lines = []
    idx = 0

    for raw in subtitles.split("\n"):
        raw = raw.strip()
        if not raw:
            continue

        lines.append(f"[{idx:04d}] {raw}")
        idx += 1

    return "\n".join(lines)

In [25]:
idx_subtitles = preprocess_subtitles(subtitles)

In [26]:
summary = llm_structured(
    instructions=instructions,
    prompt=idx_subtitles,
    output=YTSummaryResponse,
    # model='gpt-4o-mini',
    model='gemini-2.5-flash-lite'
)

summary.print()

This video introduces the AI Engineering Buildcamp, emphasizing a hands-on approach to building AI systems with best practices. The course covers a 9-week curriculum, starting with foundational concepts like Large Language Models (LLMs) and Retrieval Augmented Generation (RAG) in Week 1. Week 2 serves as a buffer for environment setup and catching up on core material, offering optional use cases. Weeks 3-6 delve into agents, testing, monitoring, and evaluation of AI systems, with a running example of building a RAG system for the Evidently AI library documentation. Weeks 7-8 are dedicated to capstone projects, including an end-to-end project example and optional advanced topics like multi-agent systems, guardrails, and deployment. The final weeks involve project demos and peer reviews, with an optional two-week hackathon for further development. The video also details communication channels like Slack, the course management platform for homework submission and peer reviews, and encoura

In [ ]:
summary.chapters[3]

Chapter(timestamp='14:03', title='Monitoring, Evaluation, and Projects', reference_line='[0317] 14:04 in week number four the focus is on')

In [ ]:
idx_subtitles.split('\n')[238:258]

['[0238] 10:32 implement it we will start talking about',
 '[0239] 10:34 agents so week number three is devoted',
 '[0240] 10:37 to agents so we will what is the',
 '[0241] 10:40 difference between uh traditional rack',
 '[0242] 10:42 and agentic rock and what agents uh what',
 '[0243] 10:46 is an agent. So we are going to cover it',
 "[0244] 10:49 here. Um we're I'm also going to",
 '[0245] 10:52 introduce one main framework that we are',
 '[0246] 10:54 going to use as uh throughout the',
 '[0247] 10:57 course. The framework is called pentki.',
 '[0248] 11:01 Um but in addition to that as bonus',
 '[0249] 11:04 materials as optional materials I will',
 '[0250] 11:06 show you other frameworks. So last',
 '[0251] 11:09 cohort we had two frameworks as like',
 '[0252] 11:12 main frameworks open AAI agents SDK and',
 '[0253] 11:15 Pentic AI and um I found it confusing at',
 '[0254] 11:19 the end. So one of them became optional',
 '[0255] 11:21 but also in addition to open agent SDK',
 "[02